In [1]:
# Files
otl = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/input_data/france_taxonlist_harmonized.tsv"
data = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/input_data/diat-barcode-taxa-harmonized.fasta"
primer_table = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/input_data/diat-barcode-primers.tsv"
run_name = 'diat-barcode-harmonized-consensus'

In [2]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
import unicodedata
import re

def clean_fasta_headers(input_file, output_file):
    """
    Cleans FASTA headers by:
    1. Removing non-ASCII characters
    2. Replacing them with ASCII equivalents where possible
    3. Removing any remaining problematic characters

    Parameters:
    input_file (str): Path to input FASTA file
    output_file (str): Path to output FASTA file

    Returns:
    int: Number of processed sequences
    """
    def clean_header(header):
        normalized = unicodedata.normalize('NFKD', header)
        ascii_text = normalized.encode('ASCII', 'ignore').decode()
        clean_text = re.sub(r'[^\w\s|.-]', '', ascii_text)
        return clean_text

    cleaned_records = []
    problematic_headers = []

    with open(input_file, 'r', encoding='utf-8') as handle:
        for record in SeqIO.parse(handle, "fasta"):
            # Clean the header
            original_header = record.description
            cleaned_header = clean_header(original_header)

            if cleaned_header != original_header:
                problematic_headers.append((original_header, cleaned_header))

            new_record = SeqRecord(
                record.seq,
                id=clean_header(record.id),
                description=cleaned_header
            )
            cleaned_records.append(new_record)

    SeqIO.write(cleaned_records, output_file, "fasta")

    print(f"Processed {len(cleaned_records)} sequences")
    if problematic_headers:
        print("\nHeaders that were modified:")
        for orig, clean in problematic_headers:
            print(f"Original: {orig}")
            print(f"Cleaned : {clean}\n")

In [3]:
clean_fasta_headers(data, data)

Processed 9233 sequences


In [4]:
from src.in_silico_analysis.amplification import InSilicoAmplification

insil = InSilicoAmplification(data, run_name=run_name)
insil.run_in_silico_analysis(primer_table)

mozaiko INFO: Checking if cutadapt is installed...
mozaiko INFO: cutadapt is installed.
mozaiko INFO: Checking if CRABS is installed...
mozaiko INFO: CRABS is installed.
mozaiko INFO: To continue the analysis, a set of primers is needed. This information should be uploaded as a TSV table and it should contain the following fields: ['target_group', 'barcode_region', 'assay_name', 'fwd_seq', 'rev_seq']
mozaiko INFO: All set. Running in-silico amplification...
mozaiko INFO: Running cutadapt command as: cutadapt -g ATGCGTTGGAGAGARCGTTTC;min_overlap=21...CAGTTGTWGGTAAATTAGAAGGTGATC;min_overlap=27 --output ../data/output_data/diat-barcode-harmonized-consensus/amplicon/rbcL_646F_998R.fasta /home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/input_data/diat-barcode-taxa-harmonized.fasta --no-indels -e 3 --revcomp --quiet --minimum-length 0 --maximum-length 552 --discard-untrimmed --action retain
mozaiko INFO: Running cutadapt command as: cutadapt -g ATGCGTTGGAGAGARCGTTTC;m

In [5]:
from src.marker_scoring.metrics_system import *

output_folder =  insil.run_dir

mex = MetricsSystemExecutor(results_folder=output_folder, otl=otl, primer_table=primer_table)
ranked_df = mex.rank_primers()

Set insert_folder_path to ../data/output_data/diat-barcode-harmonized-consensus/insert/filtered, amplicon_folder_path to ../data/output_data/diat-barcode-harmonized-consensus/amplicon/filtered and incomplete_pbs_path to ../data/output_data/diat-barcode-harmonized-consensus/incomplete_pbs/filtered/filtered_intersection.
results_folder: ../data/output_data/diat-barcode-harmonized-consensus, insert_folder_path: None, amplicon_folder_path: None
Set insert_folder_path to ../data/output_data/diat-barcode-harmonized-consensus/insert/filtered, amplicon_folder_path to ../data/output_data/diat-barcode-harmonized-consensus/amplicon/filtered and incomplete_pbs_path to ../data/output_data/diat-barcode-harmonized-consensus/incomplete_pbs/filtered/filtered_intersection.
mozaiko INFO: MultiBarcodeTools file created to ../data/output_data/diat-barcode-harmonized-consensus/insert/multibarcode/multibarcode_input.tsv
mozaiko INFO: MultiBarcodePipeline completed. Output in ../data/output_data/diat-barcode-

In [6]:
ranked_df

,primer,barcoded_taxa_one_plus,ratio_barcoded_taxa,max_mismatch_across_taxon,priming_ratio,gc_matches_across_taxon,tm_coefficient_var,tm_score,amplification_success_percent,taxonomic_resolution_percentage,...,rank_ratio_barcoded_taxa,rank_max_mismatch_across_taxon,rank_priming_ratio,rank_gc_matches_across_taxon,rank_tm_coefficient_var,rank_tm_score,rank_amplification_success_percent,rank_taxonomic_resolution_percentage,rank_divergence_score,final_rank
0,rbcL_708F1_R3_1,12.30,0.14,2273,30.17,2.0,1.19,17.16,92.18,NaN,...,1.5,4.0,3.0,4.0,4.5,4.0,5.0,NaN,4.0,35.0
1,rbcL_708F2_R3_2,12.85,0.13,2748,35.22,2.0,1.19,17.18,94.50,NaN,...,5.5,5.0,5.0,4.0,4.5,3.0,4.0,NaN,5.0,39.0
2,rbcL_708F2_R3_1,12.89,0.13,1521,55.87,2.0,1.20,17.21,94.71,39.45,...,5.5,3.0,6.0,4.0,6.5,2.0,3.0,2.0,6.0,40.0
3,rbcL_708F_DEG_R3_DEG,13.11,0.13,1161,63.95,2.0,1.20,17.02,95.17,NaN,...,5.5,2.0,7.0,4.0,6.5,6.0,2.0,NaN,7.0,41.0
4,rbcL_708F1_R3_2,12.27,0.14,3450,20.95,2.0,1.18,17.14,92.04,39.63,...,1.5,7.0,2.0,4.0,3.0,5.0,6.0,4.0,3.0,41.5
5,rbcL_708F3_R3_1,11.49,0.13,2903,30.55,2.0,1.14,16.15,90.22,NaN,...,5.5,6.0,4.0,4.0,2.0,7.0,7.0,NaN,2.0,44.5
6,rbcL_646F_998R,12.63,0.13,794,172.57,0.0,2.19,45.01,96.01,37.20,...,5.5,1.0,8.0,8.0,8.0,1.0,1.0,1.0,8.0,45.5
7,rbcL_708F3_R3_2,11.45,0.13,4035,20.80,2.0,1.13,16.11,90.21,39.56,...,5.5,8.0,1.0,4.0,1.0,8.0,8.0,3.0,1.0,47.5
